# Pose Feature Analysis

Explore the extracted pose angle features from training videos.
- Check data quality (missing values, class distribution)
- Visualize angle distributions per exercise
- See which angles separate exercises best
- Identify potentially confusing class pairs

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('tab20')

df = pd.read_csv('../data/features.csv')
print(f'Total samples: {len(df)}')
print(f'Classes: {df["label"].nunique()}')
print(f'Columns: {len(df.columns)}')
df.head()

## 1. Class Distribution

In [ ]:
# Samples per class
class_counts = df['label'].value_counts().sort_values()

fig, ax = plt.subplots(figsize=(10, 8))
class_counts.plot(kind='barh', ax=ax)
ax.set_xlabel('Number of videos')
ax.set_title('Samples per Exercise Class')
for i, v in enumerate(class_counts):
    ax.text(v + 1, i, str(v), va='center')
plt.tight_layout()
plt.show()

print(f'\nMin samples: {class_counts.min()} ({class_counts.idxmin()})')
print(f'Max samples: {class_counts.max()} ({class_counts.idxmax()})')
print(f'Mean: {class_counts.mean():.0f}, Std: {class_counts.std():.0f}')

In [ ]:
# Source distribution (btc vs crawl)
source_counts = df.groupby(['label', 'source']).size().unstack(fill_value=0)
source_counts.plot(kind='barh', stacked=True, figsize=(10, 8))
plt.xlabel('Number of videos')
plt.title('Samples per Class by Source (BTC vs Crawl)')
plt.tight_layout()
plt.show()

## 2. Feature Overview

In [ ]:
# Feature columns (exclude metadata)
feature_cols = [c for c in df.columns if c not in ['video_path', 'label', 'source']]
print(f'Feature columns ({len(feature_cols)}):')
for c in feature_cols:
    print(f'  {c}')

print(f'\nBasic stats:')
df[feature_cols].describe().round(1)

In [ ]:
# Check for missing values
missing = df[feature_cols].isnull().sum()
if missing.sum() == 0:
    print('No missing values!')
else:
    print('Missing values:')
    print(missing[missing > 0])

## 3. Angle Distributions by Exercise

Compare mean angles across exercises to see which angles best separate classes.

In [ ]:
# Mean angle per exercise
angle_mean_cols = [c for c in feature_cols if c.endswith('_mean')]

fig, axes = plt.subplots(3, 3, figsize=(18, 14))
axes = axes.flatten()

for i, col in enumerate(angle_mean_cols):
    angle_name = col.replace('_mean', '')
    ax = axes[i]
    
    # Box plot per exercise
    order = df.groupby('label')[col].median().sort_values().index
    df.boxplot(column=col, by='label', ax=ax, vert=False)
    ax.set_title(angle_name)
    ax.set_ylabel('')
    ax.set_xlabel('Degrees')

plt.suptitle('Mean Angle Distribution per Exercise', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 4. Range of Motion (Max - Min)

The difference between max and min angle shows how much each joint moves during the exercise.

In [ ]:
# Compute range of motion for each angle
angle_names = ['left_knee', 'right_knee', 'left_hip', 'right_hip',
               'left_elbow', 'right_elbow', 'left_shoulder', 'right_shoulder',
               'torso_inclination']

for name in angle_names:
    df[f'{name}_range'] = df[f'{name}_max'] - df[f'{name}_min']

range_cols = [f'{name}_range' for name in angle_names]

# Heatmap: average range of motion per exercise per angle
rom_by_exercise = df.groupby('label')[range_cols].mean()
rom_by_exercise.columns = [c.replace('_range', '') for c in rom_by_exercise.columns]

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(rom_by_exercise, annot=True, fmt='.0f', cmap='YlOrRd', ax=ax)
ax.set_title('Average Range of Motion (degrees) per Exercise')
ax.set_xlabel('Joint Angle')
ax.set_ylabel('Exercise')
plt.tight_layout()
plt.show()

## 5. Correlation Between Features

In [ ]:
# Correlation matrix of mean angles
corr = df[angle_mean_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=ax,
            xticklabels=[c.replace('_mean', '') for c in angle_mean_cols],
            yticklabels=[c.replace('_mean', '') for c in angle_mean_cols])
ax.set_title('Correlation Between Mean Angles')
plt.tight_layout()
plt.show()

## 6. Torso Inclination — Standing vs Lying vs Bent

This is a key feature for separating exercise groups.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
order = df.groupby('label')['torso_inclination_mean'].median().sort_values().index
sns.boxplot(data=df, y='label', x='torso_inclination_mean', order=order, ax=ax)
ax.set_xlabel('Torso Inclination (degrees) — 0=upright, 90=horizontal')
ax.set_ylabel('')
ax.set_title('Torso Inclination by Exercise')
ax.axvline(x=30, color='red', linestyle='--', alpha=0.5, label='30° threshold')
ax.axvline(x=60, color='orange', linestyle='--', alpha=0.5, label='60° threshold')
ax.legend()
plt.tight_layout()
plt.show()

## 7. Frames Detected per Video

Check data quality — videos with very few frames may have poor pose detection.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
df['num_frames'].hist(bins=50, ax=axes[0])
axes[0].set_xlabel('Frames with pose detected')
axes[0].set_ylabel('Count')
axes[0].set_title('Distribution of Detected Frames per Video')
axes[0].axvline(x=30, color='red', linestyle='--', label='30 frame threshold')
axes[0].legend()

# Per class
df.groupby('label')['num_frames'].median().sort_values().plot(kind='barh', ax=axes[1])
axes[1].set_xlabel('Median frames detected')
axes[1].set_title('Median Detected Frames by Exercise')

plt.tight_layout()
plt.show()

low_frames = df[df['num_frames'] < 30]
print(f'Videos with < 30 frames detected: {len(low_frames)} ({100*len(low_frames)/len(df):.1f}%)')
if len(low_frames) > 0:
    print(low_frames['label'].value_counts())

## 8. Quick Separability Check (PCA)

Project all features to 2D to see if classes form clusters.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

X = df[feature_cols].values
X_scaled = StandardScaler().fit_transform(X)

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

fig, ax = plt.subplots(figsize=(14, 10))
labels = df['label'].values
unique_labels = sorted(df['label'].unique())

for label in unique_labels:
    mask = labels == label
    ax.scatter(X_pca[mask, 0], X_pca[mask, 1], label=label, alpha=0.6, s=30)

ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
ax.set_title('PCA of Pose Features (2D projection)')
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
plt.tight_layout()
plt.show()

print(f'Variance explained by 2 components: {pca.explained_variance_ratio_.sum()*100:.1f}%')